# Non-Finite Values in grid_py

This tutorial shows how `grid_py` handles non-finite values (`NaN`,
`float('inf')`, `None`) in location and size specifications, following
the narrative of the R vignette *How grid Responds to Non-Finite Values*.

Key rules:
- **Viewports**: Non-finite locations, sizes, or scales raise an error.
- **Primitives** (lines, segments, rectangles, text, points, circles):
  non-finite positions cause the corresponding element to be silently skipped.
- **Polygons**: a `NaN` breaks the polygon into separate sub-polygons.
- **line-to**: a segment is only drawn when *both* endpoints are finite.

In [ ]:
import math

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import grid_py as gp

## Primitives with a NaN gap

We draw several primitive types at seven equally-spaced x positions.
The fourth position is set to `NaN`, so the fourth element of each
primitive should be missing.

In [ ]:
gp.grid_newpage()

x = [i / 8 for i in range(1, 8)]
x[3] = float("nan")  # 4th position is NaN

lay = gp.GridLayout(
    nrow=1, ncol=2,
    widths=gp.Unit([1, 1], ["inches", "null"]),
)
gp.push_viewport(gp.Viewport(layout=lay))
gp.grid_rect(gp=gp.Gpar(col="grey"))

# Label column
gp.push_viewport(gp.Viewport(layout_pos_col=1))
labels = ["segments", "text", "lines", "rectangles", "circles", "points"]
y_pos = [0.75, 0.6, 0.5, 0.4, 0.3, 0.2]
gp.grid_text(labels, x=1.0, y=y_pos, just="right", gp=gp.Gpar(col="grey"))
gp.pop_viewport()

# Drawing column
gp.push_viewport(gp.Viewport(layout_pos_col=2))
gp.grid_lines(x, [0.5] * 7)
gp.grid_text(list("abcde"), x, [0.6] * 5)
gp.grid_segments(x, [0.7] * 7, x, [0.8] * 7)
gp.grid_points(x, [0.2] * 7)
gp.grid_rect(x, [0.4] * 7, [0.02] * 7, [0.02] * 7)
gp.grid_circle(x, [0.3] * 7, r=[0.02] * 7)
gp.grid_text(["NA"] * 7, [0.5] * 7, [i / 10 for i in range(2, 9)],
             gp=gp.Gpar(col="grey"))
gp.pop_viewport(2)

plt.gcf()

In the output above, the fourth element of every row is absent because
its x-coordinate is `NaN`.

## Polygon and line-to with NaN breaks

For polygons, a `NaN` value splits the polygon into two separate
sub-polygons.  For `grid_line_to`, a segment is only drawn when both
the previous and current locations are finite.

In [ ]:
import numpy as np

n = 7
t = np.linspace(0, 2 * np.pi, n, endpoint=False)
cx = 0.5 + 0.4 * np.cos(t)
cy = 0.5 + 0.4 * np.sin(t)


def draw_cell(row, col, nan_indices):
    """Draw polygon + line-to inside a layout cell, with NaN at given indices."""
    gp.push_viewport(gp.Viewport(layout_pos_row=row, layout_pos_col=col))
    xv = cx.copy()
    yv = cy.copy()
    for idx in nan_indices:
        xv[idx] = float("nan")
        yv[idx] = float("nan")
    # Mark NaN positions
    for idx in nan_indices:
        gp.grid_text(
            f"NA{idx + 1}", float(cx[idx]), float(cy[idx]),
            gp=gp.Gpar(col="grey"),
        )
    # Polygon
    gp.grid_polygon(
        xv.tolist(), yv.tolist(),
        gp=gp.Gpar(fill="lightgrey", col=None),
    )
    # Line-to path (dashed)
    gp.grid_move_to(float(xv[0]), float(yv[0]))
    for i in range(1, n):
        gp.grid_line_to(
            float(xv[i]), float(yv[i]),
            gp=gp.Gpar(lty="dashed", lwd=3),
        )
    # Lines with arrow
    gp.grid_lines(
        xv.tolist(), yv.tolist(),
        arrow=gp.arrow(),
    )
    gp.grid_rect(gp=gp.Gpar(col="grey"))
    gp.pop_viewport()


gp.grid_newpage()
gp.push_viewport(
    gp.Viewport(layout=gp.GridLayout(nrow=2, ncol=2))
)

draw_cell(1, 1, [])              # No NaNs
draw_cell(1, 2, [0, n - 4])      # NaN at positions 1 and n-3
draw_cell(2, 1, [1, n - 3])      # NaN at positions 2 and n-2
draw_cell(2, 2, [2, n - 2])      # NaN at positions 3 and n-1

gp.pop_viewport()
plt.gcf()

Each panel shows the same seven circular positions, but with different
pairs made `NaN`.  The polygon splits at the gap, and dashed line-to
segments adjacent to a `NaN` are not drawn.